# Reed-Solomon Encoder

In [154]:
%load_ext autoreload
%autoreload 1
%aimport classes.galois_field
%aimport classes.galois_poly

from classes.galois_field import *
from classes.galois_poly import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 6. Generator and encoder

### Reed - Solomon (44,28). GF($2^{8}$), t=8

In [155]:
gf256  = GaloisField(8, 0b1_0001_1101)
# p(x) = x^8 + x^4 + x^3 + x^2 + 1
aux   = GaloisPoly(gf256, [GFE1])
g     = aux.poly_from_roots(list(range(16)))
print(type(g))
print(g._degree)
g

Field Closed Succesfully!, 255 Non-Zero Elements
<class 'classes.galois_poly.GaloisPoly'>
16


[0, 120, 104, 107, 109, 102, 161, 76, 3, 91, 191, 147, 169, 182, 194, 225, 120]

In [156]:
print(g)

x^16 + a^120*x^15 + a^104*x^14 + a^107*x^13 + a^109*x^12 + a^102*x^11 + a^161*x^10 + a^76*x^9 + a^3*x^8 + a^91*x^7 + a^191*x^6 + a^147*x^5 + a^169*x^4 + a^182*x^3 + a^194*x^2 + a^225*x + a^120



In [157]:
# No necesaria: el hex de g ya se imprime en la celda de comparación con el Anexo A
# print(' '.join(gf256.to_hex(g._coeff)))

In [158]:
# ISO/IEC 18004:2015, Anexo A, Tabla A.1, fila "16" (exponentes de α, de x^16 a x^0)
anexo_a = [0, 120, 104, 107, 109, 102, 161, 76, 3, 91, 191, 147, 169, 182, 194, 225, 120]
print(' '.join(gf256.to_hex(g._coeff)))
print('Coincide con Anexo A' if list(g._coeff) == anexo_a else 'NO coincide')

01 3B 0D 68 BD 44 D1 1E 08 A3 41 29 E5 62 32 24 3B
Coincide con Anexo A


In [159]:
# Verify roots
for i in range(16):
    print(i, g(i))
    if (g(i) == -1):
        print("Es raíz")
    else:
        print("No es raiz")


0 -1
Es raíz
1 -1
Es raíz
2 -1
Es raíz
3 -1
Es raíz
4 -1
Es raíz
5 -1
Es raíz
6 -1
Es raíz
7 -1
Es raíz
8 -1
Es raíz
9 -1
Es raíz
10 -1
Es raíz
11 -1
Es raíz
12 -1
Es raíz
13 -1
Es raíz
14 -1
Es raíz
15 -1
Es raíz


In [160]:
message = [f'{i:02X}' for i in range(28)]
print(message)

['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '0A', '0B', '0C', '0D', '0E', '0F', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '1A', '1B']


In [161]:
# Hex -> binario (8 bits per GF(2^8) symbol)
message_bin = [f'{int(h, 16):08b}' for h in message]
print(message_bin)

['00000000', '00000001', '00000010', '00000011', '00000100', '00000101', '00000110', '00000111', '00001000', '00001001', '00001010', '00001011', '00001100', '00001101', '00001110', '00001111', '00010000', '00010001', '00010010', '00010011', '00010100', '00010101', '00010110', '00010111', '00011000', '00011001', '00011010', '00011011']


In [162]:
packed_message = [gf256.do_pack(msg) for msg in message_bin]
print(packed_message)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]


In [163]:
msg_poly = []

for coeffs in packed_message:
    msg_poly.append(gf256._get_power(coeffs))

print(msg_poly)

[-1, 0, 1, 25, 2, 50, 26, 198, 3, 223, 51, 238, 27, 104, 199, 75, 4, 100, 224, 14, 52, 141, 239, 129, 28, 193, 105, 248]


In [164]:
msg_poly = GaloisPoly(gf256, msg_poly)
print(msg_poly)
print(' '.join(gf256.to_hex(msg_poly._coeff)))

x^26 + a^1*x^25 + a^25*x^24 + a^2*x^23 + a^50*x^22 + a^26*x^21 + a^198*x^20 + a^3*x^19 + a^223*x^18 + a^51*x^17 + a^238*x^16 + a^27*x^15 + a^104*x^14 + a^199*x^13 + a^75*x^12 + a^4*x^11 + a^100*x^10 + a^224*x^9 + a^14*x^8 + a^52*x^7 + a^141*x^6 + a^239*x^5 + a^129*x^4 + a^28*x^3 + a^193*x^2 + a^105*x + a^248

01 02 03 04 05 06 07 08 09 0A 0B 0C 0D 0E 0F 10 11 12 13 14 15 16 17 18 19 1A 1B


### Codificación sistemática RS(44,28)
p(x) = m(x)·x^16 mod g(x), con g(x) = (x-α^0)(x-α^1)···(x-α^15)  (la `g` de arriba, grado 16)

In [165]:
# x^(n-k) = x^16
x_16 = GaloisPoly(gf256, [0] + [-1]*16)
print(x_16)

x^16 + 


In [166]:
# Paridad: resto de m(x)*x^16 dividido g(x)  (ISO/IEC 18004, 7.5.2)
parity_poly = (msg_poly * x_16) % g

# Rellenar con ceros (GFE0) a la izquierda si el resto tiene grado < 15
parity = [GFE0]*(16 - len(parity_poly._coeff)) + list(parity_poly._coeff)
parity_hex = gf256.to_hex(parity)
print(' '.join(parity_hex))

# Vector de verificación
expected = 'E8 9B BB 6A 6F 00 3E 3E C9 7B C7 1E 8E 75 65 38'
print('OK' if ' '.join(parity_hex) == expected else 'NO COINCIDE')

E8 9B BB 6A 6F 00 3E 3E C9 7B C7 1E 8E 75 65 38
OK


In [167]:
# Palabra de código: mensaje (28 bytes) + paridad (16 bytes)
codeword = message + parity_hex
print(len(codeword))
print(' '.join(codeword))

44
00 01 02 03 04 05 06 07 08 09 0A 0B 0C 0D 0E 0F 10 11 12 13 14 15 16 17 18 19 1A 1B E8 9B BB 6A 6F 00 3E 3E C9 7B C7 1E 8E 75 65 38


In [168]:
N, K, N_PAR = 44, 28, 16

def rs_encode(msg_bytes):
    """Recibe K bytes (int) y devuelve los N_PAR bytes de paridad (int)."""
    m = GaloisPoly(gf256, [gf256._get_power(b) for b in msg_bytes])
    r = (m * x_16) % g
    par = [gf256(e)[ID_BIN] for e in r._coeff]   # potencias de alfa en bytes
    return [0]*(N_PAR - len(par)) + par          # completar a 16 bytes con ceros

print(' '.join(f'{b:02X}' for b in rs_encode(list(range(K)))))

E8 9B BB 6A 6F 00 3E 3E C9 7B C7 1E 8E 75 65 38


In [169]:
import numpy as np

# Paridad es el resto de
# m(x)*x^16 / g(x)  (operación lineal)

# Si se suman dos msgs, sus paridades también se suman
# paridad (m1 XOR m2) = paridad (m1) XOR paridad (m2)


# Matriz binaria 224x128: fila i = paridad del mensaje con solo el bit i en 1 (MSB primero)
G_bin = np.zeros((K*8, N_PAR*8), dtype=np.uint8)
for i in range(K*8):
    msg = [0]*K                    # List of 28 bytes
    msg[i // 8] = 1 << (7 - i % 8) # (i // 8) select byte (result = 1 each 8 steps)
                                   # (7 - i % 8) shift 1 each step
    G_bin[i] = [(b >> (7 - k)) & 1 for b in rs_encode(msg) for k in range(8)]
    # for b in rs_encode(msg)   16 bytes of parity
    # take 8 MSB & 1 LSB

print(G_bin.shape)

# Verificación: datos (bits) @ G_bin mod 2 == paridad (bits)
data_bits   = np.array([(b >> (7-k)) & 1 for b in range(K) for k in range(8)], dtype=np.uint8)
parity_bits = np.array([(int(h,16) >> (7-k)) & 1 for h in parity_hex for k in range(8)], dtype=np.uint8)
print('Matriz OK' if np.array_equal(data_bits @ G_bin % 2, parity_bits) else 'Matriz MAL')

# Unos y compuertas XOR de 2 entradas sin compartir términos
ones_per_col = G_bin.sum(axis=0)
print(f'Unos: {int(G_bin.sum())}')
print(f'XOR2: {int(np.sum(np.maximum(ones_per_col - 1, 0)))}')

(224, 128)
Matriz OK
Unos: 14330
XOR2: 14202
